In [ ]:
!pip install gym

In [ ]:
!pip install stable-baselines3

In [64]:
import numpy as np
import networkx as nx
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO

# Define the Semantic Knowledge Graph
class SemanticKnowledgeGraph:
    def __init__(self):
        self.graph = nx.DiGraph()
    
    def add_concept(self, concept):
        self.graph.add_node(concept, learned=False, accumulated_reward=0)
    
    def add_relationship(self, concept1, concept2):
        self.graph.add_edge(concept1, concept2)
    
    def get_neighbors(self, concept):
        return list(self.graph.successors(concept))
    
    def mark_learned(self, concept):
        self.graph.nodes[concept]['learned'] = True
    
    def is_learned(self, concept):
        return self.graph.nodes[concept].get('learned', False)
    
    def get_distance(self, start, end):
        try:
            return nx.shortest_path_length(self.graph, source=start, target=end)
        except nx.NetworkXNoPath:
            # Use undirected shortest path if no directed path exists
            undirected_graph = self.graph.to_undirected()
            try:
                return nx.shortest_path_length(undirected_graph, source=start, target=end)
            except nx.NetworkXNoPath:
                return float('inf')

# Define the Student
class Student:
    def __init__(self, talent_distribution, graph):
        self.knowledge_state = set()
        self.talent_distribution = talent_distribution  # Dict mapping concept to difficulty
        self.graph = graph
        self.rewards = []
    
    def query(self, current_concept, concept):
        if concept in self.knowledge_state:
            return 0  # Already learned
        distance = self.graph.get_distance(current_concept, concept)
        distance_factor = max(0.1, 1 / (distance + 1))  # Closer nodes have higher rewards
        reward = np.random.beta(2, self.talent_distribution.get(concept, 2)) * distance_factor  # Talent + distance
        self.graph.graph.nodes[concept]['accumulated_reward'] += reward
        self.rewards.append(reward)
        return reward
    
    def learn(self, concept):
        if concept not in self.knowledge_state:
            if self.graph.graph.nodes[concept]['accumulated_reward'] >= 1:  # Accumulate enough reward to learn
                self.knowledge_state.add(concept)
                self.graph.mark_learned(concept)

# Define the RL Environment
class LearningEnv(gym.Env):
    def __init__(self, graph, student):
        
        super(LearningEnv, self).__init__()

        self.graph = graph

        self.student = student
        self.current_concept =  np.random.choice(list(self.graph.graph.nodes))
        self.action_space = spaces.Discrete(len(self.graph.graph.nodes))
        self.observation_space = spaces.Discrete(len(self.graph.graph.nodes))
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        return self.action_space.sample(), {}
    
    def step(self, action):
        print("executing step with ", action)
        concepts = list(self.graph.graph.nodes)
        next_concept = concepts[action]
        reward = self.student.query(self.current_concept, next_concept)
        self.student.learn(next_concept)
        self.current_concept = next_concept
        done = all(self.graph.is_learned(c) for c in self.graph.graph.nodes)
        return action, reward, done, False, {}

# Define the Coordinator
class Coordinator:
    def __init__(self, student, graph):
        self.student = student
        self.graph = graph
        self.use_rl_strategy = False
        self.agent = PPO('MlpPolicy', env, verbose=1)
    
    #def train_agent(self, env):
     #   if self.agent is None:
            #self.agent = PPO('MlpPolicy', env, verbose=1)
      #      self.agent.learn(total_timesteps=1) # total_times is the number of steps to train the agent
    
    def switch_strategy(self):
        if (len(self.student.rewards)>10): 
            print (np.mean(self.student.rewards[-10:]))
            progress = np.mean(self.student.rewards[-10:])
            
            self.use_rl_strategy = progress < 0.05  # If progress is low, switch to RL-based strategy
    
    def give_data_to_agent(self, state, action, reward, next_state, done):
        self.agent.replay_buffer.add(state, action, reward, next_state, done)
        self.agent.train()

    def query_next_concept(self, env):
        self.switch_strategy() 
        if self.use_rl_strategy:
            #if self.agent is None:
                #self.train_agent(env)  # Train the agent if it hasn't been trained yet
            action, _ = self.agent.predict(env.action_space.sample())
        else:
            #Student has own strategy
            neighbors = self.graph.get_neighbors(env.current_concept)
            action = np.random.choice(neighbors) if neighbors else np.random.choice(list(self.graph.graph.nodes))
        return action

# Setup the Environment
graph = SemanticKnowledgeGraph()
concepts = ['Math', 'Algebra', 'Calculus', 'Physics', 'Mechanics']
for c in concepts:
    graph.add_concept(c)
graph.add_relationship('Math', 'Algebra')
graph.add_relationship('Algebra', 'Calculus')
graph.add_relationship('Physics', 'Mechanics')
print (graph.graph.nodes)
# This should be different for differently embodied students 
talent_distribution = {'Math': 2, 'Algebra': 3, 'Calculus': 5, 'Physics': 4, 'Mechanics': 6}
student = Student(talent_distribution, graph)
env = LearningEnv(graph, student)
coordinator = Coordinator(student, graph)

# Train agent while student learns
for _ in range(5):
    # Next concept to learn
    next_concept = coordinator.query_next_concept(env)
    action = concepts.index(next_concept)   
    print("Next concept to learn:", action)
    # Should execute the action in the environment 
    # how to get the index of the above action in the list of concepts
    
    
    next_state, reward, done, info, _ = env.step(action)  # Take the action in the environment
    coordinator.give_data_to_agent(env.current_concept,action,reward,next_state,done)


['Math', 'Algebra', 'Calculus', 'Physics', 'Mechanics']
Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
Next concept to learn: 4
executing step with  4


AttributeError: 'PPO' object has no attribute 'replay_buffer'

In [ ]:
np.random.beta?